# STSM Results Analysis — SW22 × TW48

Reads the result files of one STSM model (e.g. `LSTM-TA`) for all runs (A–E, In-A–In-E) and:

1. plots the seismic ratio and the anomalies of every window,
2. computes the **Z-factor** and the **average seismic ratio** against the random baseline (RST),
3. compares the model with the **baseline LSTM-AE** of the thesis.

Runs that have no result file yet are skipped.

## Setup (PC or Google Colab)

In [ ]:
import sys
import subprocess
from pathlib import Path

LOCAL_PROJECT = Path(r"D:\GIT\Attention_Model-Demeter-Seismic-Study")
DRIVE_PROJECT = Path("/content/drive/MyDrive/[FuSe][STSM] Megha Babu/Attention model/Attention_Model-Demeter-Seismic-Study")
REPO_URL = "https://github.com/Meghababu99/Attention_Model-Demeter-Seismic-Study.git"
BRANCH = "main"

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = DRIVE_PROJECT
    if not PROJECT.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(PROJECT)], check=True)
else:
    PROJECT = LOCAL_PROJECT

CODE_DIR = PROJECT / "Clean-Code"
sys.path.insert(0, str(CODE_DIR))
print("Project:", PROJECT)

## Settings

- `MODEL_NAME`: model folder in `Result Files/STSM-Results/`. The result CSVs can be in their run folders
  (as written by training) or directly in the model folder (e.g. downloaded from Drive).
- `BASELINE_NAME`: baseline to compare with.
  - `None`: the thesis results in `Result Files/SW22_TW48/` (threshold from all training pairs).
  - A model folder name, e.g. `"LSTM-AE-NormalThr"`: baseline results re-evaluated by
    `DEMETER_STSM-Evaluation-NormalThreshold.ipynb`. Use it with `MODEL_NAME = "LSTM-TA-NormalThr"`, so both use
    the threshold from normal training pairs.
- Results of this notebook are saved in `Result Files/STSM-Results/<MODEL_NAME>/Analysis_results/`.

In [ ]:
import pandas as pd

sys.path.insert(0, str(CODE_DIR / "lstm"))  # import analysis.py directly: no PyTorch needed
from analysis import (
    RUNS, rst_levels, read_results, z_factor, paired_validation, compare_with_baseline, compare_z,
    plot_ratio_per_window, plot_anomaly_bars, plot_z_factor, plot_average_ratio,
    plot_validation_comparison, plot_fewer_anomalies_higher_ratio,
)

sw, tw = 22, 48
MODEL_NAME = "LSTM-TA"
BASELINE_NAME = None

RESULTS = PROJECT / "Result Files"
MODEL_DIR = RESULTS / "STSM-Results" / MODEL_NAME
BASELINE_DIR = RESULTS / f"SW{sw}_TW{tw}"
RST_PATH = PROJECT / "Data" / f"RST_30Days_Bg_SW{sw}_TW{tw}.pkl"
OUTPUT_DIR = MODEL_DIR / "Analysis_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def model_file(run, name=MODEL_NAME):
    """Result CSV of a run: in its run folder (as written by training) or directly in the model folder (downloaded)."""
    folder = RESULTS / "STSM-Results" / name
    label = f"tw{tw}-30dBG_{run}-{name}"
    nested = folder / label / "outputs" / f"result_Hp_{label}.csv"
    return nested if nested.exists() else folder / f"result_Hp_{label}.csv"

def baseline_file(run):
    """Baseline results, 2005-2009 (and 2010 if BASELINE_NAME is set)."""
    if BASELINE_NAME:
        return model_file(run, BASELINE_NAME)
    return BASELINE_DIR / f"Retrain_result_Hp_B-tw{tw}-30dBG_{run}.csv"

def baseline_2010_file(run):
    """Thesis baseline results of 2010 (blind test): 4 windows, stored as windows 0-3, i.e. windows 16-19."""
    return BASELINE_DIR / "Blind" / f"Retrain_result_Hp_B1-tw{tw}-30dBG_{run}.csv"

## Load the results and the random baseline (RST)

The RST gives the seismic ratio expected by chance: its median (Q2) and its 0.3–99.7 % range.

In [ ]:
model = {run: read_results(model_file(run)) for run in RUNS if model_file(run).exists()}
baseline = {run: read_results(baseline_file(run)) for run in model if baseline_file(run).exists()}
rst = rst_levels(RST_PATH, tw)

print("Runs found:  ", list(model))
print("Runs missing:", [run for run in RUNS if run not in model])
print("Baseline:    ", BASELINE_NAME or f"thesis results ({BASELINE_DIR.name})", "| runs:", list(baseline))
print("RST: median {q2:.1f} %, 0.3-99.7 % range {low:.1f}-{high:.1f} %".format(**rst))

## 1. Seismic ratio and anomalies of each window

- **Ratio plot:** seismic anomaly ratio of every window (train and validation) with the RST median and range.
- **Bar plot:** anomalies of every window, split into seismic (hatched) and non-seismic.

In [ ]:
for run, df in model.items():
    plot_ratio_per_window(df, rst, f"{MODEL_NAME} — {run}", OUTPUT_DIR / f"ratio_all-{run}.png")
    plot_anomaly_bars(df, f"{MODEL_NAME} — {run}", OUTPUT_DIR / f"anomalies_all-{run}.png")

## 2. Z-factor and average seismic ratio

Over the 2005–2009 validation windows (windows 0–15; window 0 is left out for the weight-updating runs A–E):

- **Z-factor** = mean of (ratio − RST median) / error over the windows, with its 32–68 % range.
- **Average ratio** = mean ratio over the windows; its error = √(Σ error²) / number of windows.

The dashed lines mark Z = 2 and Z = 3 for 15 windows (2/√15 and 3/√15).

In [ ]:
z_model = z_factor(model, rst, tw)
z_model.to_csv(OUTPUT_DIR / f"Qfactor-SW{sw}x{sw}_TW{tw}-all_newz-{MODEL_NAME}.csv", index=False)
display(z_model.round(3))

plot_z_factor(z_model, f"Z-factor [2005-2009] — {MODEL_NAME}", n=15,
              path=OUTPUT_DIR / f"Z_va_SW{sw}_TW{tw}_all-{MODEL_NAME}.png")
plot_average_ratio(z_model, rst, f"Average Seismic Anomaly Ratio [2005-2009] — {MODEL_NAME}",
                   OUTPUT_DIR / f"Average-train_val-SW{sw}_TW{tw}-{MODEL_NAME}.png")

## 3. Comparison with the baseline LSTM-AE

The baseline is set by `BASELINE_NAME`. Both are compared on the same validation windows (0–15).

**3a. Average ratio and Z-factor:** baseline vs model, with the difference (model − baseline).

In [ ]:
z_baseline = z_factor(baseline, rst, tw)  # windows 0-15 (1-15 for A-E)
z_comparison = compare_z(z_baseline, z_model)
z_comparison.to_csv(OUTPUT_DIR / f"Baseline_vs_{MODEL_NAME}_ratio_and_Z.csv", index=False)
display(z_comparison.round(3))

**3b. Validation windows, one by one:** anomalies and seismic ratio of each window, baseline vs model.

In [ ]:
pairs = {run: paired_validation(baseline[run].query("window < 16"), model[run]) for run in baseline}

for run, paired in pairs.items():
    plot_validation_comparison(paired, MODEL_NAME, f"Model {run}", OUTPUT_DIR / f"Comparison_{run}.png")

**3c. Summary per run** (validation windows):

- total anomalies, mean ratio and ratio gain (percentage points),
- matched earthquakes (`agg_total_eq`) and seismic sequences,
- windows where the model has **fewer anomalies and a higher ratio** than the baseline.

In [ ]:
per_window, summary = compare_with_baseline(pairs)
per_window.drop(columns=["seismic_indices_base", "seismic_indices_model"]).to_csv(
    OUTPUT_DIR / f"Baseline_vs_{MODEL_NAME}_per_window.csv", index=False)
summary.to_csv(OUTPUT_DIR / f"Baseline_vs_{MODEL_NAME}_summary.csv", index=False)
display(summary.round(2))

plot_fewer_anomalies_higher_ratio(per_window, summary, MODEL_NAME,
                                  OUTPUT_DIR / f"Fewer_anomalies_higher_ratio-{MODEL_NAME}.png")

## 4. Year 2010 (windows 16–19)

Same analysis for the 2010 validation windows 16–19, for all runs:

- **Z-factor** and **average ratio** over these 4 windows; the Z = 2 and Z = 3 lines are 2/√4 and 3/√4.
- **Comparison with the baseline LSTM-AE** of 2010: windows 16–19 of `BASELINE_NAME`, or the thesis blind-test
  files (`Result Files/SW22_TW48/Blind/`) when `BASELINE_NAME = None`.

In [ ]:
W2010 = range(16, 20)

z_model_2010 = z_factor(model, rst, tw, windows=W2010)
z_model_2010.to_csv(OUTPUT_DIR / f"Qfactor-SW{sw}x{sw}_TW{tw}-all_newz-{MODEL_NAME}_2010.csv", index=False)
display(z_model_2010.round(3))

plot_z_factor(z_model_2010, f"Z-factor [2010] — {MODEL_NAME}", n=len(W2010),
              path=OUTPUT_DIR / f"Z_va_SW{sw}_TW{tw}_all-{MODEL_NAME}_2010.png")
plot_average_ratio(z_model_2010, rst, f"Average Seismic Anomaly Ratio [2010] — {MODEL_NAME}",
                   OUTPUT_DIR / f"Average-train_val-SW{sw}_TW{tw}-{MODEL_NAME}_2010.png")

In [ ]:
if BASELINE_NAME:
    baseline_2010 = {run: df.query("window >= 16") for run, df in baseline.items()}
else:
    baseline_2010 = {run: read_results(baseline_2010_file(run), first_window=16)
                     for run in model if baseline_2010_file(run).exists()}

z_comparison_2010 = compare_z(z_factor(baseline_2010, rst, tw, windows=W2010), z_model_2010)
z_comparison_2010.to_csv(OUTPUT_DIR / f"Baseline_vs_{MODEL_NAME}_ratio_and_Z_2010.csv", index=False)
display(z_comparison_2010.round(3))

pairs_2010 = {run: paired_validation(baseline_2010[run], model[run]) for run in baseline_2010}
for run, paired in pairs_2010.items():
    plot_validation_comparison(paired, MODEL_NAME, f"Model {run} — 2010", OUTPUT_DIR / f"Comparison_{run}_2010.png")

per_window_2010, summary_2010 = compare_with_baseline(pairs_2010)
per_window_2010.drop(columns=["seismic_indices_base", "seismic_indices_model"]).to_csv(
    OUTPUT_DIR / f"Baseline_vs_{MODEL_NAME}_per_window_2010.csv", index=False)
summary_2010.to_csv(OUTPUT_DIR / f"Baseline_vs_{MODEL_NAME}_summary_2010.csv", index=False)
display(summary_2010.round(2))

plot_fewer_anomalies_higher_ratio(per_window_2010, summary_2010, f"{MODEL_NAME}, 2010",
                                  OUTPUT_DIR / f"Fewer_anomalies_higher_ratio-{MODEL_NAME}_2010.png")